# fabric-rlm API tour

Run these small recipes in Microsoft Fabric with a default Lakehouse attached and write access to `/lakehouse/default/Files/fabric_rlm_examples`. They create synthetic data; no uploads are needed. Select Python 3.12 in the Fabric UI where available. `FabricLM` requires your notebook identity to have access to Fabric AI services and the configured model. Model availability depends on your workspace and region.

Run the pinned install below, **restart the Python session**, then run from the imports cell onward, not the install cell. Let pip resolve dependencies; DuckDB supports the bundled data-exploration skill. See [Fabric environment setup](../../docs/fabric-runtime-deps.md) if imports fail. Errors are not suppressed and there is no local-path fallback.

Offline tests check syntax, fixtures, and scripted runtime submissions, not live Fabric authentication, model quality, or full Fabric execution. Each `.run()` or call below uses the model and may incur capacity usage.

| Need | Recommended API |
|---|---|
| Concise input/output contract | `RLM("input -> output")` |
| Natural-language task with named inputs | `RLM.task(...)` or `RLM.from_task(...)` |
| Files in the worker | `File(...)` and `LocalArtifactStore` |
| Domain playbooks | `skills=[...]`, `list_skills()`, `SkillLoader` |
| Enforced output shape | `outputs={"field": type}` and `output_validator=` |
| Run diagnostics | `RLMResult.outputs`, `.turns`, and `.report()` |
| Fabric-hosted models | `FabricLM(...)` |
| Worker guardrails | `SecurityPolicy`, `block_network`, timeouts, and recovery |

In [ ]:
%pip install -q "fabric-rlm==0.6.4" "duckdb>=1.1"

In [ ]:
from pathlib import Path

import fabric_rlm
from fabric_rlm import (
    FabricLM,
    File,
    LocalArtifactStore,
    RLM,
    SkillLoader,
    chain,
    assert_in_range,
    list_skills,
)
from fabric_rlm.security import SecurityPolicy

print("fabric-rlm", fabric_rlm.__version__)
print("bundled skills", list_skills())

## 1. Choose a model backend

`FabricLM` uses the notebook's Fabric identity, so no API key is entered here. The model below is an example configuration, not a promise of availability. Confirm access using the [Fabric AI services model list](https://learn.microsoft.com/en-us/fabric/data-science/ai-services/ai-services-overview#consumption-rate); adjust the model and its reasoning settings together if needed.

In [ ]:
lm = FabricLM("gpt-5.1", reasoning_effort="low")

## 2. Signature API: the shortest path

Use a signature when the task is obvious from field names. Calling an `RLM` instance is shorthand for `.run(...)`.

In [ ]:
signature_rlm = RLM("numbers -> total", lm=lm, max_turns=4)
signature_result = signature_rlm(numbers=[3, 5, 8, 13])
assert signature_result.submitted, signature_result.report()
assert signature_result.outputs["total"] == 29, signature_result.outputs
signature_result.outputs

## 3. Task API: natural language, typed outputs, and validation

`RLM.task` is the recommended constructor for real work. `RLM.from_task` is the explicit alias with the same contract. A list such as `outputs=["answer"]` checks names only; a mapping enforces runtime types and gives the model repair feedback. Validators add business rules beyond type checks. For this recipe, the worker should submit a Python `int` and `float`, not numeric strings:

```python
SUBMIT(largest=int(max(sales)), average=float(sum(sales) / len(sales)))
```

`SUBMIT` is available inside generated worker code, not in the notebook's Python session.

In [ ]:
sales = [120, 80, 240, 60]
validator = chain(
    assert_in_range("largest", 0, 10_000),
    assert_in_range("average", 0, 10_000),
)

rlm = RLM.task(
    task="Use Python to return the largest sale and the arithmetic mean.",
    inputs={"sales": sales},
    outputs={"largest": int, "average": float},
    output_validator=validator,
    lm=lm,
    max_turns=5,
)

result = rlm.run()
assert result.submitted, result.report()
assert result.outputs == {"largest": 240, "average": 125.0}, result.outputs
result.outputs

Inputs can be bound in the constructor, supplied later with `rlm.run({"name": value})`, or passed through call syntax as `rlm(name=value)`. Later values override constructor-bound values, which makes one task reusable across datasets.

## 4. File inputs, artifact storage, and skills

`File` sends a lightweight path handle into the persistent worker; it does not automatically embed file contents in the prompt. Generated code can still read and print raw data, which can reach the model provider in subsequent feedback. Bounded feedback is not redacted feedback, so `File` is not a privacy boundary. `LocalArtifactStore` gives runs a predictable output directory.

Skills are Markdown playbooks that teach the model reliable domain workflows. `list_skills()` lists the bundled skills. The cell below creates the custom skill directory and writes a complete `<skill-name>.md` example, then loads it with `SkillLoader(skill_dir=...)`. Custom directories layer over the bundled skills, so both can be used in one run. A skill's verifier is guidance to generated code, not a host-enforced validator; use `output_validator=` for host enforcement.

The fixture is overwritten on each run. North has two rows totaling 500; South has 450 and West 225. Expected total revenue: 1175.0. No expected answer is passed as a task input.

In [ ]:
tour_root = Path("/lakehouse/default/Files/fabric_rlm_examples") / "api_tour"
source_dir = tour_root / "sources"
source_dir.mkdir(parents=True, exist_ok=True)
store = LocalArtifactStore(source_dir)
store.write_text(
    "regional_sales.csv",
    "region,revenue\nNorth,300\nSouth,450\nWest,225\nNorth,200\n",
)
csv_file = File(store.path("regional_sales.csv"))

skills_dir = tour_root / "skills"
skills_dir.mkdir(parents=True, exist_ok=True)
(skills_dir / "regional_sales_rules.md").write_text(
    """---
applies_when:
  keywords: [regional sales, revenue]
  output_fields: [top_region, total_revenue]
excludes: []
depends_on: []
specificity: domain
---
# Regional Sales Rules
Summary: Apply the reporting rules for the regional sales CSV.

## Purpose
Use this skill when summarizing the regional sales CSV.

## Contract: output fields
- `top_region` (`str`): region with the greatest summed revenue.
- `total_revenue` (`float`): sum of every revenue value.

## Required verifier
```python
def verify(payload):
    assert isinstance(payload.get("top_region"), str)
    assert payload["top_region"].strip()
    assert type(payload.get("total_revenue")) is float
    assert payload["total_revenue"] >= 0
```

## Tripwires
- Do not average the revenue column.
- Group by region before choosing the top region; do not choose the largest single row.
- Parse revenue as numeric before aggregation.

## Invariants
- `top_region` is non-empty.
- `total_revenue` is non-negative.

## Procedure
Read the CSV, coerce revenue to numeric, sum by region, compute both outputs, call `verify`, then submit.
""",
    encoding="utf-8",
)

skill_loader = SkillLoader(skill_dir=skills_dir)
custom_skill = skill_loader.load("regional_sales_rules")
print("available skills", skill_loader.list_skills())
print("loaded custom skill", custom_skill.title)

In [ ]:
file_result = RLM.task(
    task="Inspect the CSV. Sum revenue by region, return the region with the greatest sum and total revenue across all rows. Submit total_revenue as a Python float.",
    inputs={"sales_file": csv_file},
    outputs={"top_region": str, "total_revenue": float},
    skills=["data_exploration", "regional_sales_rules"],
    skill_loader=skill_loader,
    lm=lm,
    max_turns=6,
).run()
assert file_result.submitted, file_result.report()
assert file_result.outputs == {"top_region": "North", "total_revenue": 1175.0}, file_result.outputs
print(file_result.outputs)

## 5. Inspect the result, not only the payload

`RLMResult` records whether the model submitted, failures, token and timing totals, every executed turn, validation repairs, and a deterministic report suitable for logs or CI.

In [ ]:
print(result.report())
result.report(as_dict=True)

In [ ]:
turn_summary = [
    {
        "turn": turn.turn,
        "submitted": turn.submitted,
        "error": turn.error,
        "seconds": turn.duration_s,
    }
    for turn in result.turns
]
turn_summary

## 6. Worker controls

The default policy screens known dangerous code patterns and scrubs matching secret-like environment variables; it is not a complete security boundary. With `engine="default"`, `block_network=True` adds a Python-level socket guard, not an OS firewall: native code can bypass it, loopback remains allowed, and host-side model calls are outside its scope. It does not apply to `engine="dspy"` execution. In the default engine, timeouts are per worker execution. This recipe runs the same fixture with explicit guardrails and timeout recovery.

In [ ]:
policy = SecurityPolicy.default()
secure_rlm = RLM.task(
    task="Compute the row count without using network access.",
    inputs={"sales_file": csv_file},
    outputs={"row_count": int},
    lm=lm,
    security=policy,
    engine="default",
    block_network=True,
    timeout=120,
    recover_worker_timeouts=1,
    max_turns=4,
)
secure_result = secure_rlm.run()
assert secure_result.submitted, secure_result.report()
assert secure_result.outputs == {"row_count": 4}, secure_result.outputs
print(secure_result.outputs)

## 7. Where the rest of the API fits

- `SemanticModel` exposes a Fabric semantic model as an input when `semantic-link-sempy` is available.
- `SkillLoader` loads bundled and custom skill directories; `compose_skills` resolves dependencies.
- With `engine="default"`, worker `predict_sync(...)` and async `predict(...)` require a serializable string/dict `sub_lm` spec. Only a string/dict outer `lm` is reused implicitly; a callable outer LM such as this notebook's `FabricLM` does not configure the worker LM. Live LM objects as `sub_lm` are unsupported on this path.
- With `engine="dspy"`, nested calls use host-side `llm_query(...)` / `llm_query_batched(...)`, not worker `predict` helpers. Live `FabricLM` objects can be supplied as `lm` and `sub_lm` on that path; ask the task to use DSPy's `llm_query`.
- `verified_task(...)` compares independent runs when agreement matters more than minimum cost.
- `ReplayLM`, `ReplayInterpreter`, and `replay_trajectory(...)` support deterministic debugging.
- Excel helpers such as `add_excel_workbook_context` and `validate_target_range_sanity` support workbook-editing tasks.

For illustrative nested-LM configurations, see QUICKSTART section 7. Default-worker credentials must be configured explicitly when the scrubbed environment cannot supply them; never print/log a credential-bearing spec or bind it as a task input. Explicit `sub_lm` cannot be combined with `block_network=True`. The local `tests/test_api_arguments_engines.py::test_worker_sub_lm_roundtrip` demonstrates a serialized spec with a fake key and loopback endpoint; `tests/test_worker_predict.py` covers `predict_sync` with local stubs. Neither validates live Fabric/provider calls.

For complete options and security guidance, continue with the repository `QUICKSTART.md` and `README.md`. The focused notebooks in this directory show PDF, Spark-log, spreadsheet, and multi-source workflows.